# 00 - When To Use Deep Learning (And When Not To)

Before any framework, one question: **does this problem need a neural network at all?**

The honest answer for most business data is no. Deep learning wins decisively on data with
*structure a human can perceive* — images, audio, text — and loses to gradient boosting on
spreadsheet-shaped data more often than beginners expect.

This notebook makes you prove that to yourself with three experiments. It uses only
scikit-learn and SciPy, so it runs before you install TensorFlow or PyTorch.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cv = StratifiedKFold(5, shuffle=True, random_state=42)

## Experiment 1 — tabular data (303 rows, 13 columns)

The heart-disease dataset from module 02. This is what most real business data looks like.

In [ ]:
heart = pd.read_csv("../data/heart-disease.csv")
X_tab, y_tab = heart.drop("target", axis=1), heart["target"]

contenders = {
    "Random Forest":        RandomForestClassifier(n_estimators=300, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
    "Neural net (MLP)":     make_pipeline(StandardScaler(),
                                MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=2000,
                                              random_state=42)),
}

rows = []
for name, model in contenders.items():
    s = cross_val_score(model, X_tab, y_tab, cv=cv, scoring="roc_auc")
    rows.append({"model": name, "roc_auc": round(s.mean(), 3), "std": round(s.std(), 3)})

pd.DataFrame(rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)

Look at the ranking, and at the standard deviations. On 303 rows of tabular data, the
neural network has no advantage — and it needed scaling, more compute, and more
hyperparameters to get there.

**Why.** A neural net has to *learn* that features interact. A tree splits on interactions
directly. With only a few hundred rows there is nowhere near enough signal for the network
to discover what the tree gets for free.

## Experiment 2 — image data (1797 images, 64 pixels each)

Now the same three models on handwritten digits, where each row is an 8x8 image.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_img, y_img = digits.data, digits.target
print(f"{X_img.shape[0]} images, {X_img.shape[1]} pixels each, {len(np.unique(y_img))} classes")

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, label in zip(axes, digits.images, digits.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(label); ax.axis("off")
plt.suptitle("The raw data: 8x8 grayscale digits")
plt.show()

In [ ]:
rows = []
for name, model in contenders.items():
    s = cross_val_score(model, X_img, y_img, cv=cv, scoring="accuracy")
    rows.append({"model": name, "accuracy": round(s.mean(), 3), "std": round(s.std(), 3)})

pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)

**Read that carefully — and notice what did *not* happen.** All three are within about
half a point of each other. The neural network did not win.

This is worth dwelling on, because a lot of tutorials would quietly pick a friendlier
dataset here. The truth is that a plain fully-connected network on raw pixels has **no
built-in advantage on images**. It treats pixel 27 as just another column, exactly like the
tree does. 1797 tiny 8x8 images is nowhere near enough data for it to discover structure
on its own.

So what *does* make deep learning win on images? Not depth. **Architecture that already
knows something about images.** The next experiment shows what the raw-pixel models are
missing.

## Experiment 3 — the assumption every flat model makes

Train on the digits as they come. Then test on the *same* digits shifted one pixel down
and one pixel right. A human reads a shifted 3 as a 3 without noticing. Watch what the
models do.

In [ ]:
from scipy.ndimage import shift as nd_shift
from sklearn.model_selection import train_test_split

images = digits.images / 16.0                       # scale pixels to 0-1
X_tr_img, X_te_img, y_tr_img, y_te_img = train_test_split(
    images, y_img, test_size=0.3, random_state=42, stratify=y_img)

def shift_all(batch, dy, dx):
    return np.stack([nd_shift(im, (dy, dx), order=1, mode="constant") for im in batch])

X_te_shifted = shift_all(X_te_img, 1, 1)            # one pixel down, one pixel right

flat = lambda a: a.reshape(len(a), -1)

rows = []
for name, model in [("Random Forest", RandomForestClassifier(n_estimators=300, random_state=42)),
                    ("Neural net (MLP)", MLPClassifier((128, 64), max_iter=3000, random_state=42))]:
    model.fit(flat(X_tr_img), y_tr_img)
    rows.append({
        "model": name,
        "centred digits": round(model.score(flat(X_te_img), y_te_img), 3),
        "shifted 1 pixel": round(model.score(flat(X_te_shifted), y_te_img), 3),
    })

pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for i in range(6):
    axes[0, i].imshow(X_te_img[i], cmap="gray_r"); axes[0, i].axis("off")
    axes[1, i].imshow(X_te_shifted[i], cmap="gray_r"); axes[1, i].axis("off")
axes[0, 0].set_title("original", loc="left", fontsize=9)
axes[1, 0].set_title("shifted 1px", loc="left", fontsize=9)
plt.suptitle("Obviously the same digits - to a human")
plt.show()

**That collapse is the whole argument for deep learning on images.**

Both models go from near-perfect to worse than useless, from a shift a human would not even
notice. Neither ever learned "a 3 is a shape". They learned "pixel 27 is dark and pixel 35
is light" — a lookup table keyed on absolute position. Move the digit and the table is wrong.

A **convolutional** network cannot make that mistake, because it is built differently: it
slides the same small filter across every position, so a stroke detected in the corner is
the same feature as a stroke detected in the middle. The invariance is baked into the
architecture, not learned from data.

That is the real answer to "why deep learning for images": not more layers, but **an
architecture whose assumptions match the data**. Notebooks 05 and 07 build exactly that.

## The decision table

| Your data | Start with | Reach for deep learning when |
|---|---|---|
| Spreadsheet / tabular | **Gradient boosting** (XGBoost, LightGBM, HistGB) | almost never — and only with a lot of rows |
| Images | CNN, or a pretrained model | immediately |
| Text | a transformer (module 05) | immediately |
| Audio / video | a neural network | immediately |
| Time series | start with gradient boosting on lag features | long sequences, many related series |

### Three practical rules

1. **Rows matter more than layers.** Deep learning is data-hungry. Under ~10,000 rows on
   tabular data, gradient boosting almost always wins.
2. **Never skip the baseline.** If your network cannot beat logistic regression, the
   network is wrong — not the baseline.
3. **You will rarely train from scratch.** In practice you fine-tune a pretrained model
   (notebook 09, and module 05). Training from zero is for learning, not for shipping.

### So why learn deep learning at all?

Because images, text and audio are where the interesting problems are, and because every
LLM in module 05 is a neural network. You need the mechanics before the abstractions make
sense. That is what notebook 01 is for.

---
## Exercises

1. Increase `hidden_layer_sizes` to `(256, 128, 64)` on the tabular experiment. Does the
   MLP catch up? What does that tell you about "just add more layers"?
2. Cut the digits dataset to the first 200 samples and re-run experiment 2. Which model
   degrades most? Relate the answer to rule 1 above.
3. Time all three models on the digits data with `%%time`. Compare accuracy gained against
   time spent — would you ship the winner?
4. Add `LogisticRegression` as a fourth contender in both experiments. How close does the
   simplest possible model get?
5. **Stretch:** flatten the digit images in a scrambled but *consistent* pixel order and
   re-run. The MLP score barely moves; a CNN's would collapse. Explain why — and what that
   says about what a CNN actually assumes about its input.